# Lab 5.1 - Creating Message Passing Networks

This notebook follows the PyG tutorial "Creating Message Passing Networks".

Goal:
- understand the generic message passing formula;
- implement a custom GCN layer;
- implement an EdgeConv layer;
- solve the exercises from the end of the tutorial on the exact toy graph proposed there.

In [1]:
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn import Linear, Parameter, ReLU
from torch.nn import Sequential as Seq
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing, knn_graph, SAGEConv
from torch_geometric.nn import HeteroConv, GCNConv, GATConv, Linear as PyGLinear
from torch_geometric.utils import add_self_loops, degree

print('torch:', torch.__version__)
import torch_geometric
print('torch_geometric:', torch_geometric.__version__)

D:\Documents\FMI\Master Anul II\EDDL\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0+cu128
torch_geometric: 2.7.0


## 1. The big idea behind message passing

In PyG, many graph layers fit the same pattern:

$$x_i^{(k)} = amma^{(k)}eft(x_i^{(k-1)}, igoplus_{j n athcal{N}(i)} hi^{(k)}(x_i^{(k-1)}, x_j^{(k-1)}, e_{j,i})
ight)$$

In simple words:
- each node looks at its neighbors;
- each neighbor sends a message;
- all messages are aggregated with a permutation-invariant operation such as sum, mean or max;
- the node updates its own representation using the aggregated result.

The `MessagePassing` base class in PyG automates this pattern through `propagate()`, `message()`, `aggregate()` and `update()`.

In [2]:
class TutorialGCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.empty(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        x = self.lin(x)

        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        out = self.propagate(edge_index, x=x, norm=norm)
        out = out + self.bias
        return out

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j

In [3]:
torch.manual_seed(7)
x_demo = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 1.0]])
edge_index_demo = torch.tensor([[0, 1, 1, 2, 2, 3],
                                [1, 0, 2, 1, 3, 2]], dtype=torch.long)
conv = TutorialGCNConv(2, 3)
out_demo = conv(x_demo, edge_index_demo)
print('Input shape:', x_demo.shape)
print('Edge index shape:', edge_index_demo.shape)
print('Output shape:', out_demo.shape)
print(out_demo)

Input shape: torch.Size([4, 2])
Edge index shape: torch.Size([2, 6])
Output shape: torch.Size([4, 3])
tensor([[-0.1322,  0.1076,  0.2804],
        [-0.1849,  0.1899,  0.4203],
        [-0.2791,  0.3150,  0.6546],
        [-0.4167,  0.1830,  0.7723]], grad_fn=<AddBackward0>)


The custom GCN layer follows the tutorial exactly:
1. add self-loops so a node can also reuse its own features;
2. linearly transform node features;
3. compute degree-based normalization;
4. send normalized neighbor messages;
5. sum everything together.

In [4]:
class TutorialEdgeConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='max')
        self.mlp = Seq(
            Linear(2 * in_channels, out_channels),
            ReLU(),
            Linear(out_channels, out_channels),
        )

    def forward(self, x, edge_index):
        return self.propagate(edge_index, x=x)

    def message(self, x_i, x_j):
        tmp = torch.cat([x_i, x_j - x_i], dim=1)
        return self.mlp(tmp)


class TutorialDynamicEdgeConv(TutorialEdgeConv):
    def __init__(self, in_channels, out_channels, k=2):
        super().__init__(in_channels, out_channels)
        self.k = k

    def forward(self, x, batch=None):
        edge_index = knn_graph(x, self.k, batch=batch, loop=False, flow=self.flow)
        return super().forward(x, edge_index)

In [5]:
points = torch.tensor([[0.0, 0.0], [0.2, 0.1], [1.0, 1.1], [1.1, 0.9]], dtype=torch.float)
batch = torch.zeros(points.size(0), dtype=torch.long)
edge_conv = TutorialDynamicEdgeConv(2, 4, k=2)
edge_out = edge_conv(points, batch)
print('Points:')
print(points)
print('Dynamic EdgeConv output shape:', edge_out.shape)
print(edge_out)

Points:
tensor([[0.0000, 0.0000],
        [0.2000, 0.1000],
        [1.0000, 1.1000],
        [1.1000, 0.9000]])
Dynamic EdgeConv output shape: torch.Size([4, 4])
tensor([[ 0.7692, -0.6268, -0.3027,  0.4555],
        [ 0.7279, -0.5651, -0.3457,  0.4625],
        [ 0.5318, -0.4974, -0.4440,  0.3784],
        [ 0.5330, -0.5061, -0.4836,  0.3724]],
       grad_fn=<ScatterReduceBackward0>)


## 2. Exercises from the tutorial

The tutorial ends with a tiny graph used to reason about `GCNConv` and `EdgeConv`. Below I use that exact graph and answer each question with concrete outputs.

In [6]:
edge_index = torch.tensor([[0, 1],
                           [1, 0],
                           [1, 2],
                           [2, 1]], dtype=torch.long)
x = torch.tensor([[-1.0], [0.0], [1.0]], dtype=torch.float)
data = Data(x=x, edge_index=edge_index.t().contiguous())

print(data)
print('edge_index:')
print(data.edge_index)
print('x:')
print(data.x)

Data(x=[3, 1], edge_index=[2, 4])
edge_index:
tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])
x:
tensor([[-1.],
        [ 0.],
        [ 1.]])


In [7]:
edge_index_sl, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)
row, col = edge_index_sl
deg = degree(col, data.num_nodes, dtype=data.x.dtype)
deg_inv_sqrt = deg.pow(-0.5)
deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
x_j_identity = data.x[row]

print('edge_index with self-loops:')
print(edge_index_sl)
print('row (source nodes j):', row.tolist())
print('col (target nodes i):', col.tolist())
print('degree(col):', deg.tolist())
print('deg_inv_sqrt[row]:', deg_inv_sqrt[row].tolist())
print('deg_inv_sqrt[col]:', deg_inv_sqrt[col].tolist())
print('norm:', norm.tolist())
print('x_j when the linear map is identity:')
print(x_j_identity.squeeze(-1).tolist())

edge_index with self-loops:
tensor([[0, 1, 1, 2, 0, 1, 2],
        [1, 0, 2, 1, 0, 1, 2]])
row (source nodes j): [0, 1, 1, 2, 0, 1, 2]
col (target nodes i): [1, 0, 2, 1, 0, 1, 2]
degree(col): [2.0, 3.0, 2.0]
deg_inv_sqrt[row]: [0.7071067690849304, 0.5773502588272095, 0.5773502588272095, 0.7071067690849304, 0.7071067690849304, 0.5773502588272095, 0.7071067690849304]
deg_inv_sqrt[col]: [0.5773502588272095, 0.7071067690849304, 0.7071067690849304, 0.5773502588272095, 0.7071067690849304, 0.5773502588272095, 0.7071067690849304]
norm: [0.40824827551841736, 0.40824827551841736, 0.40824827551841736, 0.40824827551841736, 0.4999999701976776, 0.3333333134651184, 0.4999999701976776]
x_j when the linear map is identity:
[-1.0, 0.0, 0.0, 1.0, -1.0, 0.0, 1.0]


### GCN exercise answers

1. `row` stores the **source nodes** of each edge and `col` stores the **target nodes**. In the default `source_to_target` flow, messages go from `row` to `col`.
2. `degree()` counts how many edges arrive at each node index. After self-loops are added, every node contributes to its own degree as well.
3. We use `degree(col, ...)` because GCN normalization is based on the destination-side aggregation pattern. `col` tells us how many incoming messages each node receives.
4. `deg_inv_sqrt[row]` gives the source-side factor of the normalization and `deg_inv_sqrt[col]` gives the target-side factor. Multiplying them produces the symmetric GCN weight for each edge.
5. `x_j` contains the source-node features lifted to the edge level. If the linear layer is the identity map, `x_j` is literally `x[row]`, i.e. the source feature attached to every edge in the order stored by `edge_index`.

In [8]:
class GCNConvWithUpdate(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.lin_root = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.empty(out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.lin_root.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        x_msg = self.lin(x)
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        out = self.propagate(edge_index, x=x_msg, norm=norm, x_root=x)
        out = out + self.bias
        return out

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j

    def update(self, aggr_out, x_root):
        return aggr_out + self.lin_root(x_root)


torch.manual_seed(7)
base_conv = TutorialGCNConv(1, 2)
torch.manual_seed(7)
update_conv = GCNConvWithUpdate(1, 2)
print('GCN without update():')
print(base_conv(data.x, data.edge_index))
print('GCN with explicit transformed central-node features in update():')
print(update_conv(data.x, data.edge_index))

GCN without update():
tensor([[-0.1592, -0.1569],
        [ 0.0000,  0.0000],
        [ 0.1592,  0.1569]], grad_fn=<AddBackward0>)
GCN with explicit transformed central-node features in update():
tensor([[ 0.8531, -0.1845],
        [ 0.0000,  0.0000],
        [-0.8531,  0.1845]], grad_fn=<AddBackward0>)


The extra `update()` method adds a separate transformed version of the central node features after neighbor aggregation. Conceptually, this is a skip connection from the node to itself.

In [9]:
row_no_loop, col_no_loop = data.edge_index
x_i = data.x[col_no_loop]
x_j = data.x[row_no_loop]
relative = x_j - x_i
concatenated = torch.cat([x_i, relative], dim=1)

print('x_i:')
print(x_i)
print('x_j - x_i:')
print(relative)
print('torch.cat([x_i, x_j - x_i], dim=1):')
print(concatenated)
print('Resulting shape:', concatenated.shape)

x_i:
tensor([[ 0.],
        [-1.],
        [ 1.],
        [ 0.]])
x_j - x_i:
tensor([[-1.],
        [ 1.],
        [-1.],
        [ 1.]])
torch.cat([x_i, x_j - x_i], dim=1):
tensor([[ 0., -1.],
        [-1.,  1.],
        [ 1., -1.],
        [ 0.,  1.]])
Resulting shape: torch.Size([4, 2])


### EdgeConv exercise answers

1. `x_i` is the feature of the central/target node for every edge. `x_j - x_i` is the relative difference between the neighbor and the central node, so it captures how the neighbor differs from the node being updated.
2. `torch.cat([x_i, x_j - x_i], dim=1)` sticks those two pieces of information side by side along the feature dimension. We use `dim=1` because rows correspond to edges and columns correspond to features.

## Final takeaway

This tutorial shows that PyG layers are not magic black boxes. Once the message passing pattern is clear, we can implement classic operators like GCN or EdgeConv ourselves and reason about every tensor involved in propagation.